# Results Analysis

Loads CartPole benchmark CSVs from `snn_research/outputs/`, prints summary
numbers, and renders figures with Matplotlib **inline**.

**Metrics (jerk-focused):** peak jerk | endpoint error | oscillation amplitude | JAS  
_Energy metrics (spike_count, flops) have been removed._

In [ ]:
%matplotlib inline

import sys, os, glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(r"C:\Users\sriha\Downloads\SNN")
sys.path.insert(0, str(PROJECT_ROOT))

from snn_research import results
from snn_research.evaluate import jerk_accuracy_score

OUTPUTS_DIR = Path(results.OUTPUTS_DIR)
OUTPUTS_DIR.mkdir(exist_ok=True)
print("Outputs dir:", OUTPUTS_DIR)


In [ ]:
pattern = str(OUTPUTS_DIR / "*_results.csv")
files   = sorted(glob.glob(pattern))
if not files:
    raise FileNotFoundError(
        f"No result CSVs in {OUTPUTS_DIR}. Run `python -m snn_research.train` first."
    )

dfs, names = [], []
for f in files:
    df   = pd.read_csv(f)
    ctrl = df["controller"].iloc[0] if "controller" in df.columns else Path(f).stem
    dfs.append(df)
    names.append(ctrl)
    print(f"  {ctrl}: {len(df)} episodes, cols: {list(df.columns)}")


In [ ]:
METRICS = ["peak_jerk", "endpoint_err", "osc_amp"]

quintic_jerk = None
for df, n in zip(dfs, names):
    if "quintic" in n:
        quintic_jerk = df["peak_jerk"].mean()

summary_rows = []
for name, df in zip(names, dfs):
    row = {"Controller": name}
    for m in METRICS:
        if m in df.columns:
            row[f"{m}_mean"] = round(df[m].mean(), 4)
            row[f"{m}_std"]  = round(df[m].std(),  4)
    if quintic_jerk:
        for t in [0.05, 0.15]:
            jas = jerk_accuracy_score(
                row["peak_jerk_mean"], row["endpoint_err_mean"], quintic_jerk, t
            )
            row[f"JAS@{t}m"] = round(jas, 3)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df


In [ ]:
COLORS = ["#c8440a", "#1a6b5e", "#4a3d8c"]

fig, ax = plt.subplots(figsize=(8, 4))
data = [df["peak_jerk"].dropna().values for df in dfs]
bp   = ax.boxplot(data, tick_labels=names, patch_artist=True)
for patch, color in zip(bp["boxes"], COLORS):
    patch.set_facecolor(color); patch.set_alpha(0.35)
ax.set_ylabel("Peak Jerk (m/s\u00b3)")
ax.set_title("Jerk Comparison (primary metric)")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
data = [df["endpoint_err"].dropna().values for df in dfs]
bp   = ax.boxplot(data, tick_labels=names, patch_artist=True)
for patch, color in zip(bp["boxes"], COLORS):
    patch.set_facecolor(color); patch.set_alpha(0.35)
ax.set_ylabel("Endpoint Error |pos| (m)")
ax.set_title("Endpoint Accuracy (task-completion gate)")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
data = [df["osc_amp"].dropna().values for df in dfs]
bp   = ax.boxplot(data, tick_labels=names, patch_artist=True)
for patch, color in zip(bp["boxes"], COLORS):
    patch.set_facecolor(color); patch.set_alpha(0.35)
ax.set_ylabel("Oscillation Amplitude RMS (m)")
ax.set_title("Post-Stop Oscillation")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()


In [ ]:
if quintic_jerk:
    thresholds = [0.05, 0.10, 0.15, 0.25, 0.50]
    fig, ax = plt.subplots(figsize=(7, 4))
    for df, name, color in zip(dfs, names, COLORS):
        jerk_mean = df["peak_jerk"].mean()
        err_mean  = df["endpoint_err"].mean()
        jas_vals  = [jerk_accuracy_score(jerk_mean, err_mean, quintic_jerk, t)
                     for t in thresholds]
        ax.plot(thresholds, jas_vals, marker="o", label=name,
                color=color, linewidth=1.8)
    ax.axhline(1.0, color="black", linestyle="--", linewidth=0.8,
               label="Quintic baseline (JAS=1)")
    ax.set_xlabel("Endpoint error threshold (m)")
    ax.set_ylabel("JAS  (higher = better)")
    ax.set_title("Jerk-Accuracy Score vs accuracy threshold")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()
else:
    print("Quintic baseline not found.")


In [ ]:
print("=== Jerk-Accuracy Score ===")
for t in [0.05, 0.15]:
    print(f"\nerr_threshold = {t:.2f}m:")
    for df, name in zip(dfs, names):
        if quintic_jerk:
            jas = jerk_accuracy_score(
                df["peak_jerk"].mean(), df["endpoint_err"].mean(), quintic_jerk, t
            )
            if jas >= 1.0:
                verdict = "[PASS] beats baseline"
            elif jas >= 0.8:
                verdict = "[~]   near baseline"
            else:
                verdict = "[FAIL] below baseline"
            print(f"  {name:<25}  JAS={jas:.3f}  {verdict}")
